# manual-chain-forward-and-back — ex2: manually chain forward log→square→exp and run backward by hand

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `manual-chain-forward-and-back`. Running the final beacon cell reports progress against the `Backprop: manual chain forward-and-back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: manual chain forward-and-back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`manual-chain-forward-and-back`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "manual-chain-forward-and-back"
DD_SUBTOPIC = "Backprop: manual chain forward-and-back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Manual forward-and-back chain — quick refresher

The pattern at any chain length: compute forward in call order, then compute backward in REVERSE call order, threading `grad_out` through each back_fn. For a 3-step chain `a → b → c → d`:

```
# Forward:    a -> b = log(a) -> c = b**2 -> d = exp(c)
b = log(a)
c = b ** 2
d = exp(c)

# Backward (given dL/dd, want dL/da):
dL_dc = exp_back(dL_dd, d, c)       # uses cached out (d)
dL_db = square_back(dL_dc, c, b)    # uses input b: d/db b**2 = 2*b
dL_da = log_back(dL_db, b, a)       # uses input a: d/da log(a) = 1/a
```

Two invariants that hold at any length:
- **Reverse the call order.** Last forward op = first backward op.
- **Each back_fn gets `(grad_out, cached_out_at_that_node, input)`.** Which one of `out`/`input` it actually USES depends on the op.

### Exercise 2 — manually chain forward log→square→exp and run backward by hand

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the manual forward-then-backward chain pattern to a 3-operation chain: forward a → b=log(a) → c=b**2 → d=exp(c), then run backward in reverse via exp_back, square_back, log_back.
> Keywords: manual-chain, three-step, square, log, exp, reverse-order
> ```

**KCs targeted:** `manual-chain-forward-and-back`, `back-fn-uses-cached-out`

Implement `manual_chain_3(a, dL_dd)` — a hand-run forward+backward for a length-3 chain. This is the same pattern as ex1 (log+exp), stretched by one more op — a `square` middle.

**Forward pass.**
```
b = log(a)
c = b ** 2
d = exp(c)
```

**Backward pass.** Given `dL/dd`, compute `dL/da` by running the chain in REVERSE:
```
dL_dc = exp_back(dL_dd, d, c)         # d/dx exp(x) = exp(x) = d
dL_db = square_back(dL_dc, c, b)      # d/db b**2  = 2*b
dL_da = log_back(dL_db, b, a)         # d/da log(a) = 1/a
```

where:
- `exp_back(grad_out, out, x) = grad_out * out`
- `square_back(grad_out, out, x) = grad_out * 2 * x`
- `log_back(grad_out, out, x) = grad_out / x`

**Return** the 6-tuple `(b, c, d, dL_dc, dL_db, dL_da)` so the test can inspect every intermediate.

**Point of this drill.** The 2-step ex1 makes the reverse-order pattern visible; the 3-step ex2 confirms you can extend it MECHANICALLY. Each forward op picks up its mirror back_fn in the reverse pass. Assume `a > 0`.

In [ ]:
def manual_chain_3(a: Tensor, dL_dd: Tensor) -> tuple:
    # forward: a -> b -> c -> d
    b = t.log(a)
    c = b ** 2
    d = t.exp(c)
    # backward (reverse order): dL_dd -> dL_dc -> dL_db -> dL_da
    dL_dc = dL_dd * d            # exp_back: uses cached d
    dL_db = dL_dc * 2 * b        # square_back: d/db b**2 = 2*b
    dL_da = dL_db / a            # log_back: d/da log(a) = 1/a
    return b, c, d, dL_dc, dL_db, dL_da


<details><summary>Solution</summary>

```python
def manual_chain_3(a: Tensor, dL_dd: Tensor) -> tuple:
    # forward: a -> b -> c -> d
    b = t.log(a)
    c = b ** 2
    d = t.exp(c)
    # backward (reverse order): dL_dd -> dL_dc -> dL_db -> dL_da
    dL_dc = dL_dd * d            # exp_back: uses cached d
    dL_db = dL_dc * 2 * b        # square_back: d/db b**2 = 2*b
    dL_da = dL_db / a            # log_back: d/da log(a) = 1/a
    return b, c, d, dL_dc, dL_db, dL_da
```

**The pattern extends mechanically.** Every additional forward op picks up ONE additional back step in the reverse pass, threaded through `grad_out`. Once you've internalized this at length 3, length-n is the same thing in a loop — which is exactly what `backprop` will do over a sorted_graph.

**Why `square_back` uses `x` (not `out`).** `d/dx x**2 = 2x` — no relationship to `out = x**2`, so we must read `x`. Compare with `exp_back` (uses `out` because `d/dx exp(x) = out` by identity). The (`grad_out`, `out`, `x`) signature always passes both, leaving the choice to each back_fn.

**Composite identity sanity.** `d = exp((log a)**2)`, so
`d(log d)/d(log a) = 2 * log(a)` — a clean closed form that the autograd witness test confirms numerically.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()